# NBA Shot Analysis: The Evolution of Modern Basketball

An interactive Plotly dashboard exploring how NBA shot selection has evolved across 14 seasons (2011–2025), using shot-by-shot data from the top 25 scorers by points per game each year.

**Dashboard visualizations:**
1. **Scatter Plot** — shot locations across the half-court, color-coded by outcome
2. **Efficiency Histogram** — points per attempt by distance zone
3. **Line Chart** — two-point vs. three-point attempt share over time
4. **Stacked Bar Chart** — total scoring output by shot type per season

All four charts share a player dropdown and season slider.

## Setup

In [ ]:
pip install plotly pandas numpy ipywidgets

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Interactive visualizations
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Dashboard filter controls
import ipywidgets as widgets
from ipywidgets import interactive_output
from IPython.display import display

## Load and Clean Data

The raw dataset contains shot-by-shot records for the top 25 NBA players by PPG from 2011–2025, pulled from the NBA Stats API. Cleaning steps: select and rename relevant columns, parse game dates, derive a points-scored field from shot type and outcome, and reorder columns.

In [ ]:
def clean_shot_data():
    """
    Load and preprocess NBA shot chart data for the top 25 players by PPG.
    Returns a cleaned DataFrame with standardized column names and a derived Points field.
    """
    df = pd.read_csv('top_50_combined_shot_charts.csv')

    # select and rename columns
    cols = ['GAME_DATE', 'season', 'PLAYER_NAME', 'PERIOD', 'MINUTES_REMAINING',
        'SHOT_TYPE', 'EVENT_TYPE', 'SHOT_DISTANCE', 'LOC_X', 'LOC_Y']
    df = df[cols]
    df.columns = ['Game Date', 'Season', 'Player', 'Quarter', 'Minutes Remaining',
                  'Shot Type', 'Shot Result', 'Shot Distance', 'Loc X', 'Loc Y']

    # parse game date
    df['Game Date'] = pd.to_datetime(df['Game Date'], format='%Y%m%d')

    # derive points from shot type and outcome
    df['Points'] = 0
    df.loc[(df['Shot Result'] == 'Made Shot') & (df['Shot Type'] == '2PT Field Goal'), 'Points'] = 2
    df.loc[(df['Shot Result'] == 'Made Shot') & (df['Shot Type'] == '3PT Field Goal'), 'Points'] = 3

    # reorder columns
    df = df[['Game Date', 'Season', 'Player', 'Quarter', 'Minutes Remaining',
             'Shot Type', 'Shot Result', 'Points', 'Shot Distance', 'Loc X', 'Loc Y']]

    return df

In [ ]:
df = clean_shot_data()

### Dataset Overview

Here's what the first few rows look like:

In [ ]:
df.head()

### Shot Distribution Summary

In [ ]:
summary_stats = pd.DataFrame({
    'Total Shots': [len(df)],
    'Made': [(df['Shot Result'] == 'Made Shot').sum()],
    'Missed': [(df['Shot Result'] == 'Missed Shot').sum()],
    'FG%': [f"{(df['Shot Result'] == 'Made Shot').mean() * 100:.2f}%"],
    '2PA': [(df['Shot Type'] == '2PT Field Goal').sum()],
    '3PA': [(df['Shot Type'] == '3PT Field Goal').sum()],
    '3FG%': [f"{(df['Shot Type'] == '3PT Field Goal').mean() * 100:.2f}%"]
})

summary_stats

### Top Scorers in Dataset

Top 10 players by total points, with efficiency (points per shot attempt):

In [ ]:
top_player_stats = df.groupby('Player').agg({'Points': ['sum', 'count', 'mean']}).reset_index()
top_player_stats.columns = ['Player', 'Points', 'Shot Attempts', 'Efficiency']
top_player_stats = top_player_stats.nlargest(10, 'Points').reset_index(drop=True)
top_player_stats

## Visualization 1: Scatter Plot — Shot Location Analysis

Maps every shot attempt onto a half-court coordinate plane, color-coded by outcome (green = made, red = missed). Each axis carries real spatial meaning, so a scatter plot is the natural choice — court x/y coordinates place every shot in geographic context, and color encodes the third dimension (outcome) without cluttering the view.

**Key insights to look for:**
- Clusters of green near the rim and along the three-point arc (high-efficiency zones)
- The sparse mid-range region — dense attempts, lower make rate
- How individual player spatial signatures differ from the league-wide view

In [ ]:
# shared filter widgets used across all four visualizations

player_dropdown = widgets.Dropdown(
    options=['All Players'] + sorted(df['Player'].unique().tolist()),
    value='All Players',
    description='Player:',
    style={'description_width': 'initial'}
)

season_slider = widgets.SelectionSlider(
    options=['All Seasons'] + sorted(df['Season'].unique()),
    value='All Seasons',
    description='Season:',
    style={'description_width': 'initial'}
)

shot_result_dropdown = widgets.Dropdown(
    options=['Both', 'Made Shot', 'Missed Shot'],
    value='Both',
    description='Shot Result:',
    style={'description_width': 'initial'}
)

In [ ]:
def update_scatter_plot(player, season, shot_result):

    filtered_df = df.copy()

    # remove half-court heaves and out-of-bounds coordinates
    filtered_df = filtered_df[
        (filtered_df['Shot Distance'] < 35) &
        (filtered_df['Loc Y'] >= 0) &
        (filtered_df['Loc Y'] <= 470)
    ]

    # apply widget filters
    if player != 'All Players':
        filtered_df = filtered_df[filtered_df['Player'] == player]
    if season != 'All Seasons':
        filtered_df = filtered_df[filtered_df['Season'] == season]
    if shot_result != 'Both':
        filtered_df = filtered_df[filtered_df['Shot Result'] == shot_result]

    # compute title stats
    shots_taken = len(filtered_df)
    made_shots = (filtered_df['Shot Result'] == 'Made Shot').sum()
    pct_made = (made_shots / shots_taken) * 100 if shots_taken > 0 else 0

    title = f'Shot Chart: {player} - {season}<br><sub>Shots: {shots_taken:,} | FG%: {pct_made:.1f}%</sub>'

    fig = px.scatter(
        filtered_df,
        x='Loc X',
        y='Loc Y',
        color='Shot Result',
        color_discrete_map={'Made Shot': 'green', 'Missed Shot': 'red'},
        title=title,
        opacity=0.25,
        hover_data={
            'Player': True,
            'Shot Distance': ':.0f',
            'Shot Type': True,
            'Shot Result': True,
            'Loc X': False,
            'Loc Y': False
        }
    )

    fig.update_layout(
        xaxis=dict(range=[-275, 275], scaleanchor='y', showticklabels=False, showgrid=False, zeroline=False, title=''),
        yaxis=dict(range=[-25, 400], scaleanchor='x', scaleratio=1, showticklabels=False, showgrid=False, zeroline=False, title=''),
        height=600,
        width=700
    )

    fig.show()


interactive_plot = widgets.interactive(update_scatter_plot, player=player_dropdown, season=season_slider, shot_result=shot_result_dropdown)

display(interactive_plot)

## Visualization 2: Histogram — Shot Distance Efficiency

Bins shot attempts into five distance zones and calculates points per attempt for each. A red-to-green color scale makes the efficiency gradient immediately readable. This view transforms the spatial patterns in the scatter plot into a quantitative efficiency profile.

**Key insight:** The dip in the 10–16 ft zone — the mid-range inefficiency that has reshaped NBA offenses over the past decade.

In [ ]:
def update_histogram(player, season):

    filtered_df = df.copy()

    # remove half-court heaves and out-of-bounds coordinates
    filtered_df = filtered_df[
        (filtered_df['Shot Distance'] < 35) &
        (filtered_df['Loc Y'] >= 0) &
        (filtered_df['Loc Y'] <= 470)
    ]

    # apply widget filters
    if player != 'All Players':
        filtered_df = filtered_df[filtered_df['Player'] == player]
    if season != 'All Seasons':
        filtered_df = filtered_df[filtered_df['Season'] == season]

    if len(filtered_df) == 0:
        print(f'No shot data available for {player} in {season}')
        return

    # bin shots by distance
    bins = [0, 3, 10, 16, 23.75, 35]
    labels = ['0-3 ft', '3-10 ft', '10-16 ft', '16-24 ft', '24+ ft']

    filtered_df['Shot Distance Range'] = pd.cut(
        filtered_df['Shot Distance'],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    # calculate efficiency metrics per zone
    efficiency_data = filtered_df.groupby('Shot Distance Range', observed=True).agg({
        'Shot Result': lambda x: (x == 'Made Shot').mean() * 100,
        'Points': ['sum', 'count', 'mean']
    }).reset_index()

    efficiency_data.columns = ['Distance Range', 'FG%', 'Total Points', 'Attempts', 'Points Per Attempt']

    avg_ppa = efficiency_data['Points Per Attempt'].mean()
    title = f'Points Per Attempt by Distance: {player} - {season}<br><sub>Average Points Per Attempt: {avg_ppa:.2f}</sub>'

    fig = px.bar(
        efficiency_data,
        x='Distance Range',
        y='Points Per Attempt',
        title=title,
        color='Points Per Attempt',
        color_continuous_scale='RdYlGn',
        hover_data={
            'Points Per Attempt': ':.2f',
            'FG%': ':.1f',
            'Attempts': True
        }
    )

    fig.update_layout(
        xaxis=dict(title='Distance from Basket'),
        yaxis=dict(title='Points Per Attempt', range=[0, 1.5]),
        height=600,
        width=700
    )

    fig.show()


histogram_plot = widgets.interactive(update_histogram, player=player_dropdown, season=season_slider)

display(histogram_plot)

## Visualization 3: Line Chart — Evolution of Shot Selection

Tracks the percentage of two-point vs. three-point attempts across each season. Line charts are the right format for time-ordered data where the story is about trend rather than snapshot — the continuous line makes the direction and pace of change immediately readable.

**Key insight:** The sustained, league-wide shift toward three-point attempts over 14 seasons, and how individual player curves compare to that trajectory.

In [ ]:
def update_line_chart(player):

    filtered_df = df.copy()

    # remove half-court heaves and out-of-bounds coordinates
    filtered_df = filtered_df[
        (filtered_df['Shot Distance'] < 35) &
        (filtered_df['Loc Y'] >= 0) &
        (filtered_df['Loc Y'] <= 470)
    ]

    # apply widget filter
    if player != 'All Players':
        filtered_df = filtered_df[filtered_df['Player'] == player]

    # calculate shot type percentages by season
    season_data = filtered_df.groupby(['Season', 'Shot Type']).size().unstack(fill_value=0)
    season_data['Total'] = season_data.sum(axis=1)
    season_data['2PT %'] = (season_data['2PT Field Goal'] / season_data['Total']) * 100
    season_data['3PT %'] = (season_data['3PT Field Goal'] / season_data['Total']) * 100
    season_data = season_data.reset_index()

    # reshape to long format for Plotly
    season_data_melted = season_data.melt(
        id_vars=['Season'],
        value_vars=['2PT %', '3PT %'],
        var_name='Shot Type',
        value_name='Percentage'
    )

    latest_season = season_data['Season'].max()
    latest_2pt = season_data[season_data['Season'] == latest_season]['2PT %'].values[0]
    latest_3pt = season_data[season_data['Season'] == latest_season]['3PT %'].values[0]
    title = f'Shot Selection Evolution: {player}<br><sub>Latest Season ({latest_season}): {latest_2pt:.1f}% 2PT | {latest_3pt:.1f}% 3PT</sub>'

    fig = px.line(
        season_data_melted,
        x='Season',
        y='Percentage',
        color='Shot Type',
        title=title,
        markers=True,
        color_discrete_map={'2PT %': 'blue', '3PT %': 'orange'}
    )

    fig.update_layout(
        xaxis=dict(title='Season', type='category'),
        yaxis=dict(title='Percentage of Total Shots (%)', range=[0, 100]),
        height=600,
        width=700,
        hovermode='x unified'
    )

    fig.show()


line_chart = widgets.interactive(update_line_chart, player=player_dropdown)

display(line_chart)

## Visualization 4: Stacked Bar Chart — Scoring Distribution by Season

Shows total points generated from two-pointers and three-pointers in each season. This shifts the lens from attempts to actual scoring output — revealing not just how shot selection changed, but how those changes translated into points on the board.

**Key insight:** The growing share of three-point scoring in total output, and whether total points per season rise alongside the shift.

In [ ]:
def update_bar_chart(player):

    filtered_df = df.copy()

    # remove half-court heaves and out-of-bounds coordinates
    filtered_df = filtered_df[
        (filtered_df['Shot Distance'] < 35) &
        (filtered_df['Loc Y'] >= 0) &
        (filtered_df['Loc Y'] <= 470)
    ]

    # apply widget filter
    if player != 'All Players':
        filtered_df = filtered_df[filtered_df['Player'] == player]

    # aggregate total points by season and shot type
    points_by_season = filtered_df.groupby(['Season', 'Shot Type'])['Points'].sum().reset_index()
    points_by_season.columns = ['Season', 'Shot Type', 'Total Points']

    # use cleaner legend labels
    points_by_season['Shot Type'] = points_by_season['Shot Type'].map({
        '2PT Field Goal': '2-Point',
        '3PT Field Goal': '3-Point'
    })

    fig = px.bar(
        points_by_season,
        x='Season',
        y='Total Points',
        color='Shot Type',
        color_discrete_map={'2-Point': 'blue', '3-Point': 'orange'},
        title=f'Total Points by Shot Type over Time: {player}',
        barmode='stack'
    )

    fig.update_layout(
        xaxis=dict(title='Season', type='category'),
        yaxis=dict(title='Total Points'),
        height=600,
        width=700
    )

    fig.show()


bar_chart = widgets.interactive(update_bar_chart, player=player_dropdown)

display(bar_chart)

## Export Interactive Charts

Exports each visualization as a self-contained HTML file for web embedding. `include_plotlyjs='cdn'` keeps file sizes small by loading Plotly from a CDN rather than bundling it into each file.

In [ ]:
import os
os.makedirs('../exports', exist_ok=True)

WRITE_OPTS = dict(include_plotlyjs='cdn', full_html=True)

base_df = df[
    (df['Shot Distance'] < 35) & (df['Loc Y'] >= 0) & (df['Loc Y'] <= 470)
].copy()


# --- scatter plot ---
shots_taken = len(base_df)
pct_made = (base_df['Shot Result'] == 'Made Shot').mean() * 100
scatter_fig = px.scatter(
    base_df, x='Loc X', y='Loc Y', color='Shot Result',
    color_discrete_map={'Made Shot': 'green', 'Missed Shot': 'red'},
    title=f'Shot Chart: All Players - All Seasons<br><sub>Shots: {shots_taken:,} | FG%: {pct_made:.1f}%</sub>',
    opacity=0.25,
    hover_data={'Player': True, 'Shot Distance': ':.0f', 'Shot Type': True, 'Shot Result': True, 'Loc X': False, 'Loc Y': False}
)
scatter_fig.update_layout(
    xaxis=dict(range=[-275, 275], scaleanchor='y', showticklabels=False, showgrid=False, zeroline=False, title=''),
    yaxis=dict(range=[-25, 400], scaleanchor='x', scaleratio=1, showticklabels=False, showgrid=False, zeroline=False, title=''),
    height=600, width=700
)
scatter_fig.write_html('../exports/scatter_plot.html', **WRITE_OPTS)
print('Exported: scatter_plot.html')


# --- efficiency histogram ---
bins = [0, 3, 10, 16, 23.75, 35]
labels = ['0-3 ft', '3-10 ft', '10-16 ft', '16-24 ft', '24+ ft']
hist_df = base_df.copy()
hist_df['Shot Distance Range'] = pd.cut(hist_df['Shot Distance'], bins=bins, labels=labels, include_lowest=True)
efficiency_data = hist_df.groupby('Shot Distance Range', observed=True).agg({
    'Shot Result': lambda x: (x == 'Made Shot').mean() * 100,
    'Points': ['sum', 'count', 'mean']
}).reset_index()
efficiency_data.columns = ['Distance Range', 'FG%', 'Total Points', 'Attempts', 'Points Per Attempt']
avg_ppa = efficiency_data['Points Per Attempt'].mean()
hist_fig = px.bar(
    efficiency_data, x='Distance Range', y='Points Per Attempt',
    title=f'Points Per Attempt by Distance: All Players - All Seasons<br><sub>Average Points Per Attempt: {avg_ppa:.2f}</sub>',
    color='Points Per Attempt', color_continuous_scale='RdYlGn',
    hover_data={'Points Per Attempt': ':.2f', 'FG%': ':.1f', 'Attempts': True}
)
hist_fig.update_layout(xaxis=dict(title='Distance from Basket'), yaxis=dict(title='Points Per Attempt', range=[0, 1.5]), height=600, width=700)
hist_fig.write_html('../exports/histogram.html', **WRITE_OPTS)
print('Exported: histogram.html')


# --- line chart ---
season_data = base_df.groupby(['Season', 'Shot Type']).size().unstack(fill_value=0)
season_data['Total'] = season_data.sum(axis=1)
season_data['2PT %'] = (season_data['2PT Field Goal'] / season_data['Total']) * 100
season_data['3PT %'] = (season_data['3PT Field Goal'] / season_data['Total']) * 100
season_data = season_data.reset_index()
season_data_melted = season_data.melt(id_vars=['Season'], value_vars=['2PT %', '3PT %'], var_name='Shot Type', value_name='Percentage')
latest_season = season_data['Season'].max()
latest_2pt = season_data[season_data['Season'] == latest_season]['2PT %'].values[0]
latest_3pt = season_data[season_data['Season'] == latest_season]['3PT %'].values[0]
line_fig = px.line(
    season_data_melted, x='Season', y='Percentage', color='Shot Type',
    title=f'Shot Selection Evolution: All Players<br><sub>Latest Season ({latest_season}): {latest_2pt:.1f}% 2PT | {latest_3pt:.1f}% 3PT</sub>',
    markers=True, color_discrete_map={'2PT %': 'blue', '3PT %': 'orange'}
)
line_fig.update_layout(xaxis=dict(title='Season', type='category'), yaxis=dict(title='Percentage of Total Shots (%)', range=[0, 100]), height=600, width=700, hovermode='x unified')
line_fig.write_html('../exports/line_chart.html', **WRITE_OPTS)
print('Exported: line_chart.html')


# --- stacked bar chart ---
points_by_season = base_df.groupby(['Season', 'Shot Type'])['Points'].sum().reset_index()
points_by_season.columns = ['Season', 'Shot Type', 'Total Points']
points_by_season['Shot Type'] = points_by_season['Shot Type'].map({'2PT Field Goal': '2-Point', '3PT Field Goal': '3-Point'})
bar_fig = px.bar(
    points_by_season, x='Season', y='Total Points', color='Shot Type',
    color_discrete_map={'2-Point': 'blue', '3-Point': 'orange'},
    title='Total Points by Shot Type over Time: All Players',
    barmode='stack'
)
bar_fig.update_layout(xaxis=dict(title='Season', type='category'), yaxis=dict(title='Total Points'), height=600, width=700)
bar_fig.write_html('../exports/stacked_bar.html', **WRITE_OPTS)
print('Exported: stacked_bar.html')